In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: Q-CTRL Fire OpalによるQiskit Function
*[APIリファレンス](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)を参照してください*

> **Note:** Qiskit Functionsは、IBM Quantum&reg; Premium Plan、Flex Plan、およびOn-Prem（IBM Quantum Platform API経由）Planのユーザーのみが利用できる実験的な機能です。プレビューリリースの状態にあり、変更される可能性があります。


<Accordion>
<AccordionItem title="パッケージのバージョン">

このページのコードは以下の要件を使用して開発されました。
これらのバージョン以降の使用を推奨します。

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## 概要
Fire Opal Optimization Solverを使用すると、量子の専門知識を必要とせずに、量子ハードウェア上でユーティリティスケールの最適化問題を解くことができます。高レベルの問題定義を入力するだけで、Solverが残りの処理を行います。ワークフロー全体はノイズを考慮した設計になっており、内部的に[Fire OpalのPerformance Management](/guides/q-ctrl-performance-management)を活用しています。Solverは、最大規模のIBM&reg; QPU上でもフルデバイススケールで、古典的に困難な問題に対して一貫して正確な解を提供します。

Solverは柔軟性があり、目的関数または任意のグラフとして定義された組み合わせ最適化問題を解くために使用できます。問題をデバイストポロジーにマッピングする必要はありません。制約をペナルティ項として定式化できる場合、制約なしの問題と制約付きの問題の両方を解くことができます。このガイドに含まれる例では、異なるSolver入力タイプを使用して、制約なしおよび制約付きのユーティリティスケール最適化問題を解く方法を示しています。最初の例は156ノード、3-Regularグラフで定義されたMax-Cut問題に関するものであり、2番目の例はコスト関数で定義された50ノードのMinimum Vertex Cover問題に取り組んでいます。

Optimization Solverへのアクセスを取得するには、[Q-CTRLにお問い合わせください](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com)。
## 機能の説明
Solverは、ハードウェアレベルでのエラー抑制から効率的な問題マッピングおよびクローズドループの古典的最適化まで、アルゴリズム全体を完全に最適化および自動化します。Solverのパイプラインは裏側で各ステージのエラーを削減し、意味のあるスケールアップに必要な強化されたパフォーマンスを実現しています。基礎となるワークフローはQuantum Approximate Optimization Algorithm（QAOA）から着想を得ており、ハイブリッド量子古典アルゴリズムです。Optimization Solverワークフロー全体の詳細な概要については、[公開されている論文](https://arxiv.org/abs/2406.01743)を参照してください。

![Optimization Solverワークフローの視覚化](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Optimization Solverで一般的な問題を解くには：
1. 問題を目的関数、グラフ、または`SparsePauliOp`スピンチェーンとして定義します。
2. Qiskit Functions Catalogを通じて機能に接続します。
3. Solverで問題を実行し、結果を取得します。
### 受け付ける問題形式
- 目的関数の多項式表現。理想的にはPythonで既存のSymPy Polyオブジェクトを使用して作成し、[sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr)を使用して文字列にフォーマットします。
- 特定の問題タイプのグラフ表現。グラフはPythonのnetworkxライブラリを使用して作成する必要があります。その後、networkx関数`[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`を使用して文字列に変換します。
- 特定の問題のスピンチェーン表現。スピンチェーンは`SparsePauliOp`オブジェクトとして表現する必要があります。詳細については[ドキュメント](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp)を参照してください。

### サポートされているバックエンド
現在サポートされているバックエンドの一覧を表示するには、次のコードを実行してください。使用したいデバイスが一覧にない場合は、[Q-CTRLにお問い合わせ](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com)してサポートを追加してください。

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

## Example: Unconstrained optimization
Run the [maximum cut](https://en.wikipedia.org/wiki/Maximum_cut) (max-cut) problem. The following example demonstrates the Solver's capabilities on a 156-node, 3-regular unweighted graph max-cut problem, but you can also solve weighted graph problems.

In addition to `qiskit-ibm-catalog`, you will also use the following packages to run this example: `networkx` and `numpy`. You can install these packages by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [3]:
# %pip install networkx numpy

## ベンチマーク
[公開されているベンチマーク結果](https://arxiv.org/abs/2406.01743)は、Solverが120量子ビットを超える問題を正常に解き、量子アニーリングやトラップイオンデバイスで以前に公開された結果をも上回ることを示しています。以下のベンチマーク指標は、いくつかの例に基づいた問題タイプの精度とスケーリングの大まかな目安を提供します。実際の指標は、目的関数の項数（密度）とその局所性、変数の数、多項式の次数などのさまざまな問題の特性によって異なる場合があります。

「量子ビット数」は厳密な制限ではなく、非常に一貫した解の精度が期待できる大まかな閾値を表しています。これらの制限を超えた大きな問題サイズも正常に解かれており、これらの制限を超えてテストすることをお勧めします。

任意の量子ビット接続性はすべての問題タイプでサポートされています。

| 問題タイプ    | 量子ビット数 | 例 | 精度 | 合計時間 (s) | Runtime使用量 (s) | イテレーション数
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| 疎に連結された二次問題  | 156 | 3-regular Max-Cut| 100%     | 1764     | 293          | 16 |
| 高次バイナリ最適化 | 156 | Ising spin-glass model | 100%      | 1461     | 272          | 16 |
| 密に連結された二次問題 | 50 | 完全連結Max-Cut| 100%      |  1758    | 268  | 12 |
| ペナルティ項を持つ制約付き問題 | 50 | エッジ密度8%の重み付きMinimum Vertex Cover | 100%      | 1074     | 215 | 10 |
## はじめに
まず、[IBM Quantum APIキー](http://quantum.cloud.ibm.com/)を使用して認証します。次に、以下のようにQiskit Functionを選択します。（このスニペットは、すでに[アカウントをローカル環境に保存](/guides/functions#install-qiskit-functions-catalog-client)していることを前提としています。）

In [4]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [5]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.svg" alt="Output of the previous code cell" />

### 1. 問題を定義する
グラフ問題を定義し、`problem_type='maxcut'`を指定することでMax-Cut問題を実行できます。

In [6]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Run the problem
When using the graph-based input method, specify the problem type.

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [8]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 2. 問題を実行する
グラフベースの入力方法を使用する場合は、問題タイプを指定してください。

In [9]:
# Get job status
print(maxcut_job.status())

QUEUED


### 3. Retrieve the result
Retrieve the optimal cut value from the results dictionary.

<Admonition type="note">
   The mapping of the variables to the bitstring may have changed. The output dictionary contains a `variables_to_bitstring_index_map` sub-dictionary, which helps to verify the ordering.
</Admonition>

In [10]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


Qiskit Functionワークロードの[ステータス](/guides/functions#check-job-status)を確認するか、以下のように[結果](/guides/functions#retrieve-results)を返します：

In [11]:
# %pip install numpy networkx sympy

### 1. Define the problem
Define a random MVC problem by generating a graph with randomly weighted nodes.

In [12]:
import networkx as nx
from sympy import symbols, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
mvc_graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
for i in mvc_graph.nodes:
    mvc_graph.add_node(i, weight=_rng.random())

# Optionally, visualize the graph
nx.draw_networkx(mvc_graph, nx.kamada_kawai_layout(mvc_graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg" alt="Output of the previous code cell" />

### 3. 結果を取得する
結果辞書から最適なカット値を取得します。

> **Note:** 変数からビット文字列へのマッピングが変更されている可能性があります。出力辞書には`variables_to_bitstring_index_map`サブ辞書が含まれており、順序を確認するのに役立ちます。

In [13]:
# Construct the cost function.
variables = symbols([f"n[{i}]" for i in range(node_count)])
cost_function = Poly(0, variables)

for i in mvc_graph.nodes():
    weight = mvc_graph.nodes[i].get("weight", 0)
    cost_function += variables[i] * weight

Now every edge in the graph should include at least one end point from the cover, which can be expressed as the inequality:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Any case where an edge is not connected to the vertex of cover must be penalized. This can be represented in the cost function by adding a penalty of the form $P(1-n_i-n_j+n_i n_j)$ where $P$ is a positive penalty constant. Thus, an unconstrained alternative to the constrained inequality for weighted MVC is:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [14]:
# Add penalty term.
penalty_constant = 2
for i, j in mvc_graph.edges():
    cost_function += penalty_constant * (
        1 - variables[i] - variables[j] + variables[i] * variables[j]
    )

グラフが密に連結されていない場合、[PuLP](https://coin-or.github.io/pulp/)などのオープンソースソルバーを使用して問題を古典的に解くことで結果の精度を検証できます。高密度の問題では、解を検証するために高度な古典的ソルバーが必要になる場合があります。
## 例：制約付き最適化
上記のMax-Cut例は、一般的な二次制約なしバイナリ最適化問題です。Q-CTRLのOptimization Solverは、制約付き最適化を含むさまざまな問題タイプに使用できます。制約をペナルティ項としてモデル化した多項式として表された問題定義を入力することで、任意の問題タイプを解くことができます。

以下の例では、制約付き最適化問題である[minimum vertex cover](https://en.wikipedia.org/wiki/Vertex_cover)（MVC）のコスト関数を構築する方法を示します。
`qiskit-ibm-catalog`および`qiskit`パッケージに加えて、この例を実行するには`numpy`、`networkx`、および`sympy`パッケージも使用します。IPythonカーネルを使用してノートブックでこの例を実行する場合は、次のセルのコメントを解除してこれらのパッケージをインストールできます。

In [15]:
# Solve the problem
mvc_job = solver.run(
    problem=srepr(cost_function),
    backend_name=backend_name,  # E.g. "ibm_fez"
)

### 1. 問題を定義する
ランダムに重み付けされたノードを持つグラフを生成して、ランダムなMVC問題を定義します。

In [16]:
print(mvc_job.status())

QUEUED


![前のコードセルの出力](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

重み付きMVCの標準的な最適化モデルは以下のように定式化できます。まず、エッジがサブセット内の頂点に接続されていない場合、ペナルティを追加する必要があります。したがって、頂点$i$がカバー（つまりサブセット）に含まれる場合は$n_i = 1$、そうでない場合は$n_i = 0$とします。次に、目標はサブセット内の頂点の総数を最小化することであり、以下の関数で表すことができます：

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [17]:
mvc_result = mvc_job.result()
qctrl_cost = mvc_result["solution_bitstring_cost"]

# Print results
print(f"Solution cost: {qctrl_cost}")

Solution cost: 10.248198273708624


グラフ内のすべてのエッジは、カバーから少なくとも1つのエンドポイントを含む必要があり、以下の不等式として表すことができます：

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

エッジがカバーの頂点に接続されていないすべての場合は、ペナルティを科す必要があります。これは、$P$が正のペナルティ定数である$P(1-n_i-n_j+n_i n_j)$の形式のペナルティを追加することでコスト関数に表すことができます。したがって、重み付きMVCの制約付き不等式に対する制約なしの代替案は次のとおりです：

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$